# 14. Diffusion and flow training targets — DDPM, Flow Matching, Rectified Flow reflow, MeanFlow

Tensor dimension and training budget are reduced. The target definitions and the **actual Rectified Flow reflow stage** are preserved.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import jvp

torch.manual_seed(7)
device = torch.device("cpu")
print("device:", device)

## 1. Diffusion forward path and epsilon / x0 / v parameterizations

In [ ]:
x0 = torch.tensor([[1.0, -1.0]], device=device)
epsilon = torch.tensor([[0.5, 2.0]], device=device)
alpha_t = torch.tensor([[0.8]], device=device)
sigma_t = torch.sqrt(1 - alpha_t.square())

x_t = alpha_t * x0 + sigma_t * epsilon

target_epsilon = epsilon
target_x0 = x0
target_v = alpha_t * epsilon - sigma_t * x0

print("x_t:", x_t)
print("epsilon/x0/v:", target_epsilon, target_x0, target_v)

## 2. Flow Matching straight conditional path

In [ ]:
data = torch.tensor([[1.0, -1.0]], device=device)
noise = torch.tensor([[-0.5, 1.5]], device=device)
t = torch.tensor([0.3], device=device)

z_t = (1 - t[:, None]) * data + t[:, None] * noise
instantaneous_velocity = noise - data

print("FM point:", z_t)
print("FM target velocity:", instantaneous_velocity)

## 3. Rectified Flow — first flow, generated coupling, then reflow

Rectified Flow is not just the straight target formula. Reflow uses the learned first flow to generate a new endpoint coupling and trains a second flow on those re-coupled endpoints.

In [ ]:
class VelocityField(nn.Module):
    def __init__(self, data_dim=2, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, x, t):
        return self.net(
            torch.cat([x, t[:, None]], dim=-1)
        )


def rectified_flow_loss(model, source, destination):
    batch_size = source.size(0)
    t = torch.rand(batch_size, device=source.device)
    x_t = (
        (1 - t[:, None]) * source
        + t[:, None] * destination
    )
    target = destination - source
    prediction = model(x_t, t)
    return F.mse_loss(prediction, target)


@torch.no_grad()
def euler_flow(model, source, steps=8):
    x = source.clone()
    step_size = 1.0 / steps

    for step_index in range(steps):
        t_value = step_index / steps
        t = torch.full(
            (source.size(0),),
            t_value,
            device=source.device,
        )
        x = x + step_size * model(x, t)
    return x


# Initial independent source/target coupling.
source = torch.randn(24, 2, device=device)
target = torch.randn(24, 2, device=device) + torch.tensor(
    [1.0, -0.5],
    device=device,
)

first_flow = VelocityField().to(device)
first_optimizer = torch.optim.Adam(first_flow.parameters(), lr=3e-3)
for _ in range(8):
    first_optimizer.zero_grad()
    loss = rectified_flow_loss(first_flow, source, target)
    loss.backward()
    first_optimizer.step()

# Reflow coupling: transport each source sample through the learned first flow.
reflow_destination = euler_flow(first_flow, source, steps=8).detach()

second_flow = VelocityField().to(device)
second_optimizer = torch.optim.Adam(second_flow.parameters(), lr=3e-3)
for _ in range(8):
    second_optimizer.zero_grad()
    reflow_loss = rectified_flow_loss(
        second_flow,
        source,
        reflow_destination,
    )
    reflow_loss.backward()
    second_optimizer.step()

assert reflow_destination.shape == source.shape
print("first-flow loss:", loss.item())
print("reflow loss:", reflow_loss.item())
print("reflow pairs:", reflow_destination.shape)

## 4. MeanFlow average-velocity identity with JVP

MeanFlow directly models `u(z_t,r,t)`. The target uses
`u = v - (t-r) d_t u`, where the total derivative follows tangent `(v, 0, 1)` through `(z,r,t)`. The target is stop-gradient.

In [ ]:
class MeanVelocityModel(nn.Module):
    def __init__(self, data_dim=2, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, z, r, t):
        condition = torch.cat(
            [z, r[:, None], t[:, None]],
            dim=-1,
        )
        return self.net(condition)


meanflow = MeanVelocityModel().to(device)
meanflow_optimizer = torch.optim.Adam(meanflow.parameters(), lr=3e-3)

for step in range(5):
    meanflow_optimizer.zero_grad()

    data = torch.randn(16, 2, device=device)
    noise = torch.randn_like(data)
    t = torch.rand(16, device=device)
    r = torch.rand(16, device=device) * t

    z_t = (1 - t[:, None]) * data + t[:, None] * noise
    v = noise - data

    prediction, total_derivative = jvp(
        lambda z, r_value, t_value: meanflow(
            z,
            r_value,
            t_value,
        ),
        (z_t, r, t),
        (
            v,
            torch.zeros_like(r),
            torch.ones_like(t),
        ),
    )
    target = (
        v
        - (t - r)[:, None] * total_derivative
    ).detach()

    meanflow_loss = F.mse_loss(prediction, target)
    meanflow_loss.backward()
    meanflow_optimizer.step()

print("MeanFlow loss:", meanflow_loss.item())

## 5. Diagonal consistency check: MeanFlow → Flow Matching at r=t

In [ ]:
data = torch.randn(8, 2, device=device)
noise = torch.randn_like(data)
t = torch.rand(8, device=device)
r = t.clone()
z_t = (1 - t[:, None]) * data + t[:, None] * noise
v = noise - data

prediction, total_derivative = jvp(
    lambda z, r_value, t_value: meanflow(z, r_value, t_value),
    (z_t, r, t),
    (v, torch.zeros_like(r), torch.ones_like(t)),
)
diagonal_target = v - (t - r)[:, None] * total_derivative

assert torch.allclose(diagonal_target, v)
print("max diagonal error:", (diagonal_target - v).abs().max().item())

## Structural checklist

The notebook now executes the missing Rectified Flow **reflow coupling and second training stage**, while MeanFlow uses the JVP/stop-gradient objective rather than an analytic stand-in average.